# ResidualCascade Hybrid — Comprehensive Training (2x T4)

## Kiến trúc
```

```

In [ ]:
!pip install -q sentence-transformers pyarrow pandas faiss-cpu tqdm
import os, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import faiss
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
base_lo = Path('/kaggle/input/datasets/nguynteo/lastone/amazon_recs_output')
base_lr = Path('/kaggle/input/datasets/nguynteo/lastreview/amazon_recs_output')
dfs = [pq.read_table(f).to_pandas() for f in sorted(base_lo.glob('test/*.parquet'))]
inter = pd.concat(dfs, ignore_index=True)
u_uniq,u_inv = np.unique(inter['user_idx'],return_inverse=True)
i_uniq,i_inv = np.unique(inter['item_idx'],return_inverse=True)
U,I = len(u_uniq),len(i_uniq)
inter['u']=u_inv; inter['i']=i_inv
rng=np.random.default_rng(123); msk=rng.random(len(inter))<0.8
tr=inter[msk]; te=inter[~msk]
val_msk = rng.random(len(tr)) < 0.1
val_df = tr[val_msk].copy()
tr = tr[~val_msk].copy()
u2items_tr={int(u):set(g['i'].values) for u,g in tr.groupby('u')}
print('U,I',U,I,'train',len(tr),'val',len(val_df),'test',len(te))

In [ ]:
# MF-bias
class MF(nn.Module):
    def __init__(self,n_u,n_i,f=64):
        super().__init__()
        self.ue=nn.Embedding(n_u,f); self.ie=nn.Embedding(n_i,f)
        self.ub=nn.Embedding(n_u,1); self.ib=nn.Embedding(n_i,1)
        self.g=nn.Parameter(torch.tensor(0.0))
        nn.init.normal_(self.ue.weight,std=0.01); nn.init.normal_(self.ie.weight,std=0.01)
    def forward(self,u,i):
        return (self.ue(u)*self.ie(i)).sum(1)+self.ub(u).squeeze()+self.ib(i).squeeze()+self.g
dev=torch.device('cuda')
mf=MF(U,I,64)
if torch.cuda.device_count()>1: mf=nn.DataParallel(mf)
mf=mf.to(dev)
opt=torch.optim.Adam(mf.parameters(),lr=0.005,weight_decay=1e-6)
lossf=nn.MSELoss()
from torch.utils.data import TensorDataset, DataLoader as DL
dl_mf=DL(TensorDataset(torch.from_numpy(tr['u'].values.astype(np.int64)),torch.from_numpy(tr['i'].values.astype(np.int64)),torch.from_numpy(tr['rating'].values.astype(np.float32))),batch_size=32768,shuffle=True,num_workers=4,pin_memory=True)
for ep in range(5):
    mf.train();tl=0;t0=time.time()
    pbar=tqdm(dl_mf,desc=f'MF Epoch {ep+1}/5')
    for ub,ib,rb in pbar:
        ub,ib,rb=ub.to(dev),ib.to(dev),rb.to(dev)
        opt.zero_grad();p=mf(ub,ib);l=lossf(p,rb);l.backward();opt.step();tl+=l.item()
        pbar.set_postfix({'rmse': f'{(tl/(pbar.n*32768+1))**0.5:.4f}'})
    print(f'MF Epoch {ep+1}/5: RMSE={(tl/len(tr))**0.5:.4f} | {time.time()-t0:.1f}s', flush=True)
mf.eval()
m=mf.module if hasattr(mf,'module') else mf
mf_ue=m.ue.weight.detach().cpu().numpy(); mf_ie=m.ie.weight.detach().cpu().numpy()
print(f'MF done: {mf_ue.shape}, {mf_ie.shape}')
np.save('/kaggle/working/mf_ue.npy', mf_ue)
np.save('/kaggle/working/mf_ie.npy', mf_ie)

In [ ]:
lr_files = sorted(base_lr.glob('test_with_text/*.parquet'))
lr_df = pd.concat([pq.read_table(f).to_pandas() for f in lr_files], ignore_index=True)
agg = lr_df.groupby('item_idx')['review_content'].apply(lambda x:' . '.join(map(str,x.tolist()[:3]))).reset_index()
i2row={int(x):i for i,x in enumerate(i_uniq)}
itxt={i2row[int(r['item_idx'])]:str(r['review_content'])[:300] for _,r in agg.iterrows() if int(r['item_idx']) in i2row}
sbert=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device='cuda')
emb_c=np.zeros((I,384),dtype=np.float32)
keys=list(itxt.keys()); txts=[itxt[k] for k in keys]
if txts:
    ec=sbert.encode(txts,batch_size=512,show_progress_bar=True,convert_to_numpy=True,normalize_embeddings=True)
    for k,ii in enumerate(keys): emb_c[ii]=ec[k]
user_prof_all=np.zeros((U,384),dtype=np.float32)
for u,its in u2items_tr.items():
    vs=[emb_c[i] for i in its if np.linalg.norm(emb_c[i])>0]
    if vs:
        v=np.mean(vs,axis=0); v/= (np.linalg.norm(v)+1e-8); user_prof_all[u]=v
print('SBERT done', emb_c.shape)
np.save('/kaggle/working/emb_c.npy',emb_c)
np.save('/kaggle/working/user_prof_all.npy',user_prof_all)

In [ ]:
def make_feat_batch(us, is_):
    mf_u=mf_ue[us]; mf_i=mf_ie[is_]
    cb_u=user_prof_all[us]; cb_i=emb_c[is_]
    return np.concatenate([mf_u,mf_i,cb_u,cb_i],axis=1).astype(np.float32)

class BPRDS(Dataset):
    def __init__(self,tr_df,u2set,n_items,neg=4,hard=True):
        self.u=tr_df['u'].values.astype(np.int64); self.i=tr_df['i'].values.astype(np.int64)
        self.u2set=u2set; self.n_items=n_items; self.neg=neg; self.hard=hard
    def __len__(self): return len(self.u)
    def __getitem__(self,idx):
        u=int(self.u[idx]); i=int(self.i[idx])
        neg=[]; seen=self.u2set.get(u,set())
        if self.hard:
            if np.linalg.norm(emb_c[i])>0:
                pe=(emb_c[i]/np.linalg.norm(emb_c[i])).reshape(1,-1).astype(np.float32)
                import faiss
                valid_items=np.array([x for x in range(I) if np.linalg.norm(emb_c[x])>0])
                valid_embs=emb_c[valid_items].astype(np.float32)
                norms=np.linalg.norm(valid_embs,axis=1,keepdims=True); norms[norms==0]=1
                idx_faiss=faiss.IndexFlatIP(valid_embs.shape[1]); idx_faiss.add(valid_embs/norms)
                D,II=idx_faiss.search(pe,200)
                hard_cands=[int(valid_items[x]) for x in II[0] if int(valid_items[x]) not in seen and int(valid_items[x])!=i]
                neg=hard_cands[:self.neg]
            while len(neg)<self.neg:
                j=np.random.randint(0,self.n_items)
                if j not in seen and j!=i and j not in neg: neg.append(j)
        else:
            while len(neg)<self.neg:
                j=np.random.randint(0,self.n_items)
                if j not in seen and j!=i: neg.append(j)
        return u,i,np.array(neg,dtype=np.int64)

u2items_tr_set={int(u):set(g['i'].values) for u,g in tr.groupby('u')}
ds=BPRDS(tr,u2items_tr_set,I,neg=4,hard=True)
dl=DataLoader(ds,batch_size=2048,shuffle=True,num_workers=4,pin_memory=True)
dev=torch.device('cuda')
print(f'Dataset: {len(ds):,} samples (hard negatives)')


In [ ]:
class ResidualCascade(nn.Module):
    def __init__(self, in_dim=896, hidden=128, drop=0.2):
        super().__init__()
        self.coarse = nn.Sequential(nn.Linear(in_dim,64), nn.ReLU(), nn.Linear(64,1))
        self.fine = nn.Sequential(nn.Linear(in_dim+1, hidden), nn.ReLU(), nn.Dropout(drop), nn.Linear(hidden, 64), nn.ReLU(), nn.Linear(64,1))
    def forward(self, x):
        c = self.coarse(x)
        return (self.fine(torch.cat([x, c], dim=1)) + c).squeeze()

In [ ]:
model = ResidualCascade()
model = model.to(dev)
if torch.cuda.device_count()>1: model = nn.DataParallel(model)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=15)
t0=time.time()
best_val_hr10=0; patience=3; no_improve=0; history=[]

def mm(x):
    lo,hi=x.min(),x.max()
    if hi-lo<1e-12: return np.ones_like(x)*0.5
    return (x-lo)/(hi-lo)

def eval_split(split_df, K_list=[5,10,20], nn=99, n_users=300):
    model.eval()
    h={f'HR@{k}':0 for k in K_list}
    nd={f'NDCG@{k}':0 for k in K_list}
    mrr_sum=0; n=0
    rng=np.random.default_rng(888)
    su=rng.choice(split_df['u'].unique(), size=min(n_users,len(split_df['u'].unique())), replace=False)
    for u in tqdm(su, desc='Eval', leave=False):
        pu=split_df[split_df['u']==int(u)]['i'].values
        if len(pu)==0: continue
        pos=int(pu[0]); seen=set(u2items_tr.get(int(u),set()))|{pos}
        negs=[]
        while len(negs)<nn:
            x=int(rng.integers(0,I))
            if x!=pos and x not in seen: negs.append(x)
        if len(negs)<nn: continue
        cands=np.array([pos]+negs)
        X=make_feat_batch(np.array([int(u)]*len(cands)), cands)
        xt=torch.from_numpy(X).to(dev)
        with torch.no_grad():
            if hasattr(model,'module'): sc=model.module(xt).cpu().numpy()
            else: sc=model(xt).cpu().numpy()
        order=np.argsort(sc)[::-1]; rp=int(np.where(order==0)[0][0]); n+=1
        for K in K_list:
            if rp<K:
                h[f'HR@{K}']+=1; nd[f'NDCG@{K}']+=1.0/np.log2(rp+2)
        mrr_sum+=1.0/(rp+1)
    for K in K_list:
        h[f'HR@{K}']/=n; nd[f'NDCG@{K}']/=n
    return {'HR@5':h['HR@5'],'HR@10':h['HR@10'],'HR@20':h['HR@20'],
            'NDCG@5':nd['NDCG@5'],'NDCG@10':nd['NDCG@10'],'NDCG@20':nd['NDCG@20'],
            'MRR':mrr_sum/n, 'n':n}

for ep in range(15):
    model.train(); tl=0; n=0
    pbar = tqdm(dl, desc=f'Epoch {ep+1}/15')
    for u,i,neg in pbar:
        u=u.numpy() if torch.is_tensor(u) else np.array(u)
        i=i.numpy() if torch.is_tensor(i) else np.array(i)
        neg=neg.numpy() if torch.is_tensor(neg) else np.array(neg)
        B=u.shape[0]; Nn=neg.shape[1]
        Xpos=make_feat_batch(u,i)
        neg_flat=neg.reshape(-1)
        u_rep=np.repeat(u,Nn)
        Xneg=make_feat_batch(u_rep,neg_flat)
        xt=torch.from_numpy(Xpos).to(dev); xn=torch.from_numpy(Xneg).to(dev)
        if hasattr(model,'module'):
            pos=model.module(xt); negs=model.module(xn)
        else:
            pos=model(xt); negs=model(xn)
        loss=-torch.log(torch.sigmoid(pos.repeat_interleave(Nn)-negs)+1e-12).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        tl+=loss.item()*B; n+=B
        pbar.set_postfix({'loss': f'{tl/n:.4f}'})
    sched.step()
    train_loss = tl/n
    val_metrics = eval_split(val_df, n_users=300)
    val_hr10 = val_metrics['HR@10']
    print(f'Epoch {ep+1}/15: loss={train_loss:.4f} val_HR@10={val_hr10:.4f} val_NDCG@10={val_metrics["NDCG@10"]:.4f} val_MRR={val_metrics["MRR"]:.4f} time={time.time()-t0:.1f}s', flush=True)
    history.append({'epoch':ep+1,'train_loss':train_loss,'val':val_metrics})
    if val_hr10 > best_val_hr10:
        best_val_hr10 = val_hr10
        no_improve = 0
        sd = model.module.state_dict() if hasattr(model,'module') else model.state_dict()
        torch.save(sd, '/kaggle/working/best_model.pt')
        print(f'  -> New best! Saved best_model.pt')
    else:
        no_improve += 1
        if no_improve >= 3:
            print(f'  Early stopping at epoch {ep+1}')
            break

print('\n===== TEST SET EVALUATION =====')
test_metrics = eval_split(te, n_users=1000)
print(json.dumps(test_metrics, indent=2))
torch.save(model.module.state_dict() if hasattr(model,'module') else model.state_dict(), '/kaggle/working/final_model.pt')
open('/kaggle/working/eval_results.json','w').write(json.dumps({'history':history,'test':test_metrics},indent=2))
print('SAVED: best_model.pt, final_model.pt, eval_results.json')


In [ ]:
# Save model
sd = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
torch.save(sd, f'/kaggle/working/{MODEL_NAME}_model.pt')
print(f'{MODEL_NAME} saved')
